# MLflow Tracking on Your Local Machine

This notebook mirrors the updated README. It shows how to train a small PyTorch model on your own machine, log metrics and artifacts to a local MLflow server, register the model, and load it back for inference.

## Setup

Before running the cells below:

1. Create and activate a virtual environment.
2. Install PyTorch for your machine (CPU-only or GPU build), plus `mlflow`, `jupyter`, and `ipykernel`.
3. Start the MLflow server in another terminal from this project directory:

```bash
mlflow server --host 127.0.0.1 --port 5000 --backend-store-uri sqlite:///mlflow.db --default-artifact-root ./artifacts
```

4. Open `http://127.0.0.1:5000` to view the MLflow UI while this notebook runs.

In [ ]:
import os
import time

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import mlflow
import mlflow.pytorch
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
EXPERIMENT_NAME = "TinyLinearRegression"
MODEL_NAME = "TinyLinearRegressionModel"

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
client = MlflowClient(tracking_uri=TRACKING_URI)

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    raise RuntimeError("MLflow experiment was not created. Start the server and rerun this cell.")

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment ID:", experiment.experiment_id)

In [ ]:
torch.manual_seed(7)

x = torch.linspace(-2, 2, 256).unsqueeze(1)
y = 3.0 * x - 0.5 + 0.2 * torch.randn_like(x)

dataset = TensorDataset(x, y)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

print("x shape:", x.shape)
print("y shape:", y.shape)

In [ ]:
class TinyRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)

    def forward(self, x):
        return self.linear(x)


def train_model(model, loader, epochs=30, lr=0.05):
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    history = []

    for epoch in range(epochs):
        total_loss = 0.0
        for xb, yb in loader:
            optimizer.zero_grad()
            preds = model(xb)
            loss = loss_fn(preds, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(xb)

        epoch_loss = total_loss / len(loader.dataset)
        history.append(epoch_loss)

    return history

In [ ]:
model = TinyRegressor()
params = {
    "epochs": 30,
    "learning_rate": 0.05,
    "batch_size": 32,
    "optimizer": "SGD",
}
example_input = x[:4]

with mlflow.start_run(run_name="local_notebook_run") as run:
    mlflow.log_params(params)

    losses = train_model(
        model,
        loader,
        epochs=params["epochs"],
        lr=params["learning_rate"],
    )

    for step, loss in enumerate(losses):
        mlflow.log_metric("train_loss", loss, step=step)

    signature = infer_signature(
        example_input.numpy(),
        model(example_input).detach().numpy(),
    )
    logged_model = mlflow.pytorch.log_model(
        model,
        name="model",
        signature=signature,
        input_example=example_input,
    )
    mlflow.log_text(f"Final train_loss={losses[-1]:.4f}\n", "summary.txt")

    run_id = run.info.run_id

print("Run ID:", run_id)
print("Final loss:", losses[-1])

## Inspect the run in MLflow

Refresh the UI at `http://127.0.0.1:5000` to compare the logged parameters, metrics, and artifacts. The next cell shows the same information from Python.

In [ ]:
runs = client.search_runs(
    [experiment.experiment_id],
    order_by=["attributes.start_time DESC"],
    max_results=5,
)

for item in runs:
    print("run_id=", item.info.run_id)
    print("  status=", item.info.status)
    print("  params=", item.data.params)
    print("  metrics=", item.data.metrics)
    print()

In [ ]:
model_uri = logged_model.model_uri
registered = mlflow.register_model(model_uri, MODEL_NAME)

version_info = None
for _ in range(30):
    version_info = client.get_model_version(MODEL_NAME, registered.version)
    if version_info.status == "READY":
        break
    time.sleep(1)

print("Registered version:", registered.version)
print("Registry status:", version_info.status)

client.transition_model_version_stage(
    name=MODEL_NAME,
    version=registered.version,
    stage="Staging",
)
client.transition_model_version_stage(
    name=MODEL_NAME,
    version=registered.version,
    stage="Production",
)

version_info = client.get_model_version(MODEL_NAME, registered.version)
print("Current stage:", version_info.current_stage)

In [ ]:
production_model = mlflow.pytorch.load_model(f"models:/{MODEL_NAME}/Production")

test_x = torch.tensor([[1.5], [-0.5]])
with torch.no_grad():
    predictions = production_model(test_x)

print("Inputs:")
print(test_x)
print()
print("Predictions:")
print(predictions)